# N2 — Convencionalización de metáforas (Visualización)
## AI-MELT: Nivel 2 — Exploración visual

**Prerequisito:** Haber ejecutado `N2_convencionalizacion.ipynb` (procesamiento).

---

### Archivos de entrada esperados (en `./outputs/N2/`)

| Archivo | Columnas clave | Descripción |
|---|---|---|
| `n2_metaforas_convencionales.csv` | ID_metaconvencional, metafora_conceptual, dominio_fuente, dominio_meta, enfoque_origen, frecuencia_absoluta, robustez, dispersion, cluster_id | Metáforas convencionales con robustez |
| `n2_primaria_convencional.csv` | ID_metaconvencional, ID_expresion | Tabla M:N primarias ↔ convencionales |
| `n2_embeddings_cache.npz` | mc_embeddings, df_embeddings, sim_matrix, linkage_matrix | Embeddings y matrices precalculados |
| `n2_artifacts.json` | unique_mc, unique_df, labels_*, comparison, inertias, silhouettes_km | Labels de clustering y métricas |

### Archivos de N1 también usados

| Archivo | Columnas clave |
|---|---|
| `n1_metaforas_primarias.csv` | ID_expresion, metafora_conceptual, dominio_fuente, dominio_meta, ID_documento, volumen, capitulo |

---

### Visualizaciones generadas
1. Top 15 metáforas convencionales por frecuencia
2. Dendrograma de clustering jerárquico
3. UMAP 2D de dominios fuente coloreado por cluster
4. Comparación de algoritmos (silhouette + elbow)
5. Heatmap de distribución textual (metáfora × volumen/capítulo)
6. Sankey: metáforas primarias → metáforas convencionales

## 1. Configuración

In [ ]:
# ============================================================
# 1. IMPORTS Y CARGA DE DATOS
# ============================================================
# Descomenta si necesitas instalar:
# !pip install matplotlib seaborn plotly umap-learn pandas

import gc
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

# Rutas
N1_DIR = "./outputs/N1/"
N2_DIR = "./outputs/N2/"
VIZ_DIR = "./outputs/N2/viz/"
os.makedirs(VIZ_DIR, exist_ok=True)

# --- Cargar datos ---
df_conv = pd.read_csv(os.path.join(N2_DIR, "n2_metaforas_convencionales.csv"))
df_mn = pd.read_csv(os.path.join(N2_DIR, "n2_primaria_convencional.csv"))
df_n1 = pd.read_csv(os.path.join(N1_DIR, "n1_metaforas_primarias.csv"))

cache = np.load(os.path.join(N2_DIR, "n2_embeddings_cache.npz"), allow_pickle=True)
mc_embeddings = cache["mc_embeddings"]
df_embeddings = cache["df_embeddings"]
linkage_matrix = cache["linkage_matrix"]

with open(os.path.join(N2_DIR, "n2_artifacts.json")) as f:
    artifacts = json.load(f)

unique_mc = artifacts["unique_mc"]
unique_df = artifacts["unique_df"]
labels_dbscan = np.array(artifacts["labels_dbscan"])
labels_kmeans = np.array(artifacts["labels_kmeans"])
labels_hierarchical = np.array(artifacts["labels_hierarchical"])
best_algo = artifacts["best_algo"]

print("✓ Datos cargados:")
print(f"  Metáforas convencionales: {len(df_conv)}")
print(f"  Relaciones M:N: {len(df_mn):,}")
print(f"  Metáforas primarias N1: {len(df_n1):,}")
print(f"  Embeddings MC: {mc_embeddings.shape}")
print(f"  Embeddings DF: {df_embeddings.shape}")

## 2. Top 15 metáforas convencionales por frecuencia

In [ ]:
# ============================================================
# 2. TOP 15 POR FRECUENCIA
# ============================================================

df_freq = df_conv[df_conv["enfoque_origen"] == "frecuencia"].sort_values("frecuencia_absoluta", ascending=True)
top15 = df_freq.tail(15)

fig, ax = plt.subplots(figsize=(12, 8))
colors = {"ALTA": "#1D9E75", "MODERADA": "#E8A838", "DÉBIL": "#D85A30"}
bar_colors = [colors.get(r, "#888") for r in top15["robustez"]]

ax.barh(range(len(top15)), top15["frecuencia_absoluta"].values, color=bar_colors)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels([mc[:60] for mc in top15["metafora_conceptual"]], fontsize=8)
ax.set_xlabel("Frecuencia absoluta")
ax.set_title("Top 15 metáforas convencionales por frecuencia")

# Leyenda
from matplotlib.patches import Patch

legend_elements = [Patch(facecolor=c, label=l) for l, c in colors.items()]
ax.legend(handles=legend_elements, title="Robustez", loc="lower right")

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "viz_top15_frecuencia.png"), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 3. Dendrograma de clustering jerárquico

In [ ]:
# ============================================================
# 3. DENDROGRAMA
# ============================================================
from scipy.cluster.hierarchy import dendrogram

fig, ax = plt.subplots(figsize=(16, 8))

# Limitar labels si hay muchos dominios
max_labels = 60
labels_display = [d[:25] for d in unique_df[:max_labels]] if len(unique_df) > max_labels else [d[:25] for d in unique_df]

dendro = dendrogram(
    linkage_matrix,
    labels=labels_display if len(unique_df) <= max_labels else None,
    leaf_rotation=90,
    leaf_font_size=7,
    color_threshold=1.5,
    ax=ax,
    truncate_mode='lastp' if len(unique_df) > max_labels else None,
    p=max_labels if len(unique_df) > max_labels else None,
)

ax.set_title(f"Dendrograma de dominios fuente (clustering jerárquico, {len(unique_df)} dominios)")
ax.set_ylabel("Distancia")
ax.axhline(y=1.5, color='red', linestyle='--', alpha=0.5, label='Corte')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "viz_dendrograma.png"), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 4. UMAP 2D de dominios fuente coloreado por cluster

In [ ]:
# ============================================================
# 4. UMAP 2D
# ============================================================
try:
    import umap

    # Seleccionar labels del mejor algoritmo
    label_map = {"DBSCAN": labels_dbscan, "K-means": labels_kmeans, "Jerárquico": labels_hierarchical}
    selected_labels = label_map[best_algo]

    reducer = umap.UMAP(n_components=2, random_state=42, metric="cosine", n_neighbors=min(15, len(df_embeddings)-1))
    coords = reducer.fit_transform(df_embeddings)

    fig, ax = plt.subplots(figsize=(14, 10))
    n_clusters = len(set(selected_labels)) - (1 if -1 in selected_labels else 0)
    cmap = plt.cm.get_cmap("tab20", max(n_clusters, 1))

    for label in sorted(set(selected_labels)):
        mask = selected_labels == label
        color = "lightgray" if label == -1 else cmap(label % 20)
        lbl = "Outliers" if label == -1 else f"Cluster {label}"
        ax.scatter(coords[mask, 0], coords[mask, 1], c=[color], label=lbl, s=60, alpha=0.7, edgecolors='white')

        # Anotar nombres de dominios
        for j in np.where(mask)[0]:
            ax.annotate(unique_df[j][:20], (coords[j, 0], coords[j, 1]),
                         fontsize=6, alpha=0.7, ha='center', va='bottom')

    ax.set_title(f"UMAP 2D — Dominios fuente por cluster ({best_algo})")
    ax.legend(fontsize=7, loc='best', ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_umap_clusters.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

except ImportError:
    print("⚠ umap-learn no instalado. Ejecuta: pip install umap-learn")

## 5. Comparación de algoritmos de clustering

In [ ]:
# ============================================================
# 5. SILHOUETTE + ELBOW
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow method (K-means)
valid_ks = artifacts["valid_ks"]
inertias = artifacts["inertias"]
silhouettes_km = artifacts["silhouettes_km"]

axes[0].plot(valid_ks, inertias, 'bo-', markersize=5)
best_k = artifacts["best_k"]
axes[0].axvline(x=best_k, color='red', linestyle='--', label=f'Mejor k={best_k}')
axes[0].set_xlabel("k (número de clusters)")
axes[0].set_ylabel("Inercia")
axes[0].set_title("Elbow Method (K-means)")
axes[0].legend()

# Silhouette comparison
algo_names = list(artifacts["comparison"].keys())
sil_values = [artifacts["comparison"][a]["silhouette"] or 0 for a in algo_names]
colors = ["#3B8BD4", "#1D9E75", "#D85A30"]
axes[1].bar(algo_names, sil_values, color=colors)
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Comparación de algoritmos")
for j, (name, val) in enumerate(zip(algo_names, sil_values, strict=False)):
    axes[1].text(j, val + 0.01, f"{val:.3f}", ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "viz_comparacion_algoritmos.png"), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 6. Heatmap de distribución textual (metáfora × volumen)

In [ ]:
# ============================================================
# 6. HEATMAP: METÁFORA CONVENCIONAL × VOLUMEN
# ============================================================

# Unir convencionales con primarias y luego con metadatos N1
df_joined = df_mn.merge(df_n1[["ID_expresion", "ID_documento"]].drop_duplicates(), on="ID_expresion", how="left")

# Intentar obtener volumen
if "volumen" in df_n1.columns:
    vol_map = df_n1[["ID_documento", "volumen"]].drop_duplicates().set_index("ID_documento")["volumen"]
    df_joined["volumen"] = df_joined["ID_documento"].map(vol_map)
    group_col = "volumen"
else:
    group_col = "ID_documento"

# Contar: metáfora convencional × volumen
df_joined = df_joined.merge(df_conv[["ID_metaconvencional", "metafora_conceptual"]].drop_duplicates(),
                              on="ID_metaconvencional", how="left")

# Top 20 metáforas para legibilidad
top20_mc = df_conv.sort_values("frecuencia_absoluta", ascending=False).head(20)["metafora_conceptual"].tolist()
df_heat = df_joined[df_joined["metafora_conceptual"].isin(top20_mc)]

if len(df_heat) > 0 and group_col in df_heat.columns:
    cross = pd.crosstab(df_heat["metafora_conceptual"], df_heat[group_col])
    # Reordenar por frecuencia total
    cross = cross.loc[cross.sum(axis=1).sort_values(ascending=False).index]
    # Truncar labels
    cross.index = [mc[:50] for mc in cross.index]
    if group_col == "volumen":
        cross.columns = [v[:25] for v in cross.columns]

    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(cross, annot=True, fmt="d", cmap="YlOrRd", ax=ax, linewidths=0.5)
    ax.set_title(f"Distribución textual: metáfora convencional × {group_col} (top 20)")
    ax.set_xlabel(group_col.capitalize())
    ax.set_ylabel("Metáfora convencional")
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_heatmap_distribucion.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
else:
    print("⚠ No hay datos suficientes para el heatmap de distribución.")

del df_joined, df_heat
gc.collect()

## 7. Sankey: metáforas primarias → metáforas convencionales

In [ ]:
# ============================================================
# 7. SANKEY: PRIMARIAS → CONVENCIONALES (Plotly)
# ============================================================

# Tomar top 20 metáforas convencionales y sus primarias
top20_ids = df_conv.sort_values("frecuencia_absoluta", ascending=False).head(20)["ID_metaconvencional"].tolist()
df_sankey = df_mn[df_mn["ID_metaconvencional"].isin(top20_ids)]

# Contar primarias por convencional
sankey_counts = df_sankey.groupby("ID_metaconvencional").size().reset_index(name="count")
sankey_counts = sankey_counts.merge(
    df_conv[["ID_metaconvencional", "metafora_conceptual", "dominio_fuente", "dominio_meta", "enfoque_origen"]],
    on="ID_metaconvencional", how="left"
)

# Dominios meta únicos (izquierda) → metáforas convencionales (derecha)
metas = sorted(sankey_counts["dominio_meta"].dropna().unique().tolist())
mcs = sankey_counts["metafora_conceptual"].tolist()

all_labels = [f"[META] {m}" for m in metas] + [mc[:45] for mc in mcs]
node_index = {lbl: i for i, lbl in enumerate(all_labels)}

sources, targets, values, colors = [], [], [], []
color_map = {"frecuencia": "#3B8BD488", "clustering": "#7F77DD88"}

for _, row in sankey_counts.iterrows():
    meta_label = f"[META] {row['dominio_meta']}"
    mc_label = row["metafora_conceptual"][:45]
    if meta_label in node_index and mc_label in node_index:
        sources.append(node_index[meta_label])
        targets.append(node_index[mc_label])
        values.append(int(row["count"]))
        colors.append(color_map.get(row["enfoque_origen"], "#88888888"))

if sources:
    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=10, thickness=15, line=dict(color="black", width=0.5),
                   label=all_labels),
        link=dict(source=sources, target=targets, value=values, color=colors)
    )])
    fig.update_layout(
        title="Sankey: dominio META → metáforas convencionales (top 20, color=enfoque)",
        font_size=10, height=700
    )
    out_path = os.path.join(VIZ_DIR, "viz_sankey_convencionales.html")
    pio.write_html(fig, out_path, include_plotlyjs="cdn")
    print(f"✓ {out_path}")
    fig.show()
else:
    print("⚠ Sin datos para Sankey.")

## 8. Resumen

In [ ]:
# ============================================================
# 8. RESUMEN
# ============================================================

print("=" * 60)
print("N2 — VISUALIZACIONES COMPLETADAS")
print("=" * 60)
print(f"\nArchivos en {VIZ_DIR}:")
for f_name in sorted(os.listdir(VIZ_DIR)):
    size = os.path.getsize(os.path.join(VIZ_DIR, f_name)) / 1024
    icon = "📊" if f_name.endswith(".html") else "🖼"
    print(f"  {icon} {f_name} ({size:.0f} KB)")
print("\n➡ SIGUIENTE: Ejecutar N3_escenarios.ipynb")